In [1]:
import requests
import pandas as pd
import json
import os
import numpy as np
from time import sleep
from scipy.stats import norm
import warnings

# Configurações de exibição e avisos
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100
warnings.filterwarnings('ignore')


In [2]:
# ---------------------------------------------------
# 1. PARÂMETROS E PREÇOS TETO
# ---------------------------------------------------
SELIC = 0.1475  # Taxa livre de risco padronizada para 2026

precos_teto_suno_dividendos = {
    "WIZC3": 10.00, "BBSE3": 35.50, "BBAS3": 25.00, "UNIP6": 70.00, "SEER3": 14.00, "VALE3": 75.00,
    "PETR4": 34.00, "AXIA6": 43.80, "TUPY3": 21.00, "AGRO3": 27.50, "EGIE3": 28.60, "ITSA4": 9.50,
}

carteira_PM = {
    "ABEV3": 10.00, "B3SA3": 11.08, "BBAS3": 21.96, "BBSE3": 32.00,
    "EGIE3": 28, "FLRY3": 15.60, "HYPE3": 29.31, "ITSA4": 9.41,
    "KLBN11": 18.58, "LEVE3": 33.92, "PETR4": 30.06, "TAEE11": 38.28,
    "UNIP6": 50.99, "VALE3": 58.13, "RADL3": 25
}

precos_teto = {k: precos_teto_suno_dividendos.get(k, v) for k, v in carteira_PM.items()}

In [ ]:
# ---------------------------------------------------
# 2. DOWNLOAD E ATUALIZAÇÃO (CORREÇÃO JSON)
# ---------------------------------------------------
if not os.path.exists("dbJson"):
    os.makedirs("dbJson")

def atualizarDados(ativo):
    url = f"https://storage.googleapis.com/api-cdn-eaglesystem/api/{ativo.upper()}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return f"{ativo}: erro HTTP {response.status_code}"
        
        # Validação de conteúdo JSON para evitar erro 'Expecting value'
        content = response.text.strip()
        if not (content.startswith('{') or content.startswith('[')):
            return f"{ativo}: Erro - Conteúdo não é JSON válido"

        dados = response.json()
        with open(f"dbJson/{ativo}.json", "w", encoding="utf-8") as arq:
            json.dump(dados, arq, indent=2)
        return f"{ativo}: atualizado"
    except Exception as e:
        return f"{ativo}: erro -> {e}"

for ativo in precos_teto.keys():
    print(atualizarDados(ativo))


In [3]:
# ---------------------------------------------------
# 3. PROCESSAMENTO DOS DADOS (DATAFRAME)
# ---------------------------------------------------
def dataFrameUnico(ativo):
    with open(f"dbJson/{ativo}.json", "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    preco_atual = dados["asset"]["close"]
    linhas = []

    for serie in dados["series"]:
        vencimento = serie.get("due_date")
        dias = serie.get("days_to_maturity")

        for strike in serie["strikes"]:
            strike_price = strike["strike"]
            for tipo in ["call", "put"]:
                opt = strike[tipo]
                if opt:
                    linhas.append({
                        "ativo": ativo, "tipo": tipo.upper(), "vencimento": vencimento,
                        "dias": dias, "strike": strike_price, "symbol": opt["symbol"],
                        "bid": opt["bid"], "ask": opt["ask"], "volume": opt["volume"],
                        "delta": opt["bs"]["delta"], "theta": opt["bs"]["theta"],
                        "vol": opt["bs"]["volatility"], "poe": opt["bs"]["poe"],
                        "preco_atual": preco_atual
                    })
    return pd.DataFrame(linhas)

todos = []
for ativo, pteto in precos_teto.items():
    try:
        df = dataFrameUnico(ativo)
        df["preco_teto"] = pteto
        todos.append(df)
    except: pass

df_final = pd.concat(todos, ignore_index=True)

In [6]:
# ---------------------------------------------------
# 4. CÁLCULOS E FILTROS DE PUTS
# ---------------------------------------------------
puts = df_final[df_final["tipo"] == "PUT"].copy()
puts = puts[puts['vol'] > 0].copy() # Evita erro matemático

puts["retorno"] = (puts["bid"] / (puts["strike"] - puts['bid'])) * 100
puts["retorno_mes"] = puts["retorno"] * (30 / puts["dias"].replace(0, 1))
puts["dist_strike"] = ((puts["strike"] / puts["preco_atual"]) - 1) * 100

# Black-Scholes para PUTS
T = (puts['dias'] / 252).replace(0, 0.001)
sigma = puts['vol'] / 100
d1 = (np.log(puts['preco_atual'] / puts['strike']) + (SELIC + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
puts['black_scholes'] = (puts['strike'] * np.exp(-SELIC * T) * norm.cdf(-d2) - puts['preco_atual'] * norm.cdf(-d1)).round(2)
puts['desvio_bs'] = (puts['bid'] - puts['black_scholes']).round(2)

aporte = 5000
puts['cotas'] = np.floor((aporte / puts['strike'])/100) * 100
puts['premio X cotas'] = puts['cotas'] * puts['bid']

# Ranking e Score
filtro_put = puts[

    (puts['dias'].between(1, 90)) & 
    (puts['dist_strike'] <= -10) & 
    (puts['retorno_mes'] >= 1) & 
    (puts['delta'] >= -0.3) &
    (puts['volume'] > 0)

].copy()

if not filtro_put.empty:
    filtro_put['score'] = (
        filtro_put['dist_strike'].rank(ascending=True) * 5 + 
        filtro_put['retorno_mes'].rank(ascending=False) * 4
    )
    print("\n--- MELHORES OPÇÕES DE VENDA DE PUT ---")
    
    display(filtro_put[

        [
        'ativo','symbol', 'tipo', 'strike', 'preco_atual',  
        'preco_teto', 'dist_strike', 'bid', 'ask', 'black_scholes',
        'volume', 'delta', 'theta', 'vol', 'poe',  'retorno', 'retorno_mes',
        'desvio_bs', 'dias', 'vencimento','score', 'cotas', 'premio X cotas'
        ]
        
        ].sort_values('score').head(20))

    


--- MELHORES OPÇÕES DE VENDA DE PUT ---


,ativo,symbol,tipo,strike,preco_atual,preco_teto,dist_strike,bid,ask,black_scholes,volume,delta,theta,vol,poe,retorno,retorno_mes,desvio_bs,dias,vencimento,score,cotas,premio X cotas
1645,B3SA3,B3SAP161,PUT,16.06,18.51,11.08,-13.236089,0.06,0.00,0.09,107100,-0.048518,-0.008899,60.208,5.84,0.375000,1.250000,-0.03,9,2026-04-17,40.0,300.0,18.0
2053,B3SA3,B3SAQ162,PUT,16.22,18.51,11.08,-12.371691,0.23,0.00,0.25,54600,-0.154966,-0.011065,47.970,19.56,1.438399,1.598221,-0.02,27,2026-05-15,42.0,300.0,69.0
1641,B3SA3,B3SAP156,PUT,15.56,18.51,11.08,-15.937331,0.05,0.00,0.06,69900,-0.022433,-0.004754,63.340,2.78,0.322373,1.074576,-0.01,9,2026-04-17,45.0,300.0,15.0
10621,PETR4,PETRQ40,PUT,40.50,48.10,34.00,-15.800416,0.39,0.58,0.43,53200,-0.067132,-0.013174,49.469,8.59,0.972326,1.080362,-0.04,27,2026-05-15,46.0,100.0,39.0
9687,PETR4,PETRP419,PUT,41.90,48.10,34.00,-12.889813,0.15,0.35,0.20,1637700,-0.026860,-0.011781,56.601,3.19,0.359281,1.197605,-0.05,9,2026-04-17,49.0,100.0,15.0
1649,B3SA3,B3SAP166,PUT,16.56,18.51,11.08,-10.534846,0.11,0.32,0.15,68400,-0.092831,-0.014536,59.223,10.89,0.668693,2.228977,-0.04,9,2026-04-17,59.0,300.0,33.0
1643,B3SA3,B3SAP158,PUT,15.81,18.51,11.08,-14.586710,0.05,0.00,0.07,68900,-0.033529,-0.006621,61.029,4.09,0.317259,1.057530,-0.02,9,2026-04-17,59.0,300.0,15.0
9693,PETR4,PETRP426,PUT,42.65,48.10,34.00,-11.330561,0.20,48.57,0.25,163200,-0.045040,-0.017887,54.460,5.27,0.471143,1.570475,-0.05,9,2026-04-17,66.0,100.0,20.0
9697,PETR4,PETRP431,PUT,43.15,48.10,34.00,-10.291060,0.21,0.80,0.29,305100,-0.061637,-0.022824,53.019,7.14,0.489054,1.630182,-0.08,9,2026-04-17,73.0,100.0,21.0
9691,PETR4,PETRP424,PUT,42.40,48.10,34.00,-11.850312,0.15,0.45,0.23,248000,-0.038152,-0.015673,55.038,4.49,0.355030,1.183432,-0.08,9,2026-04-17,73.0,100.0,15.0


In [5]:
# ---------------------------------------------------
# 5. CÁLCULOS E FILTROS DE CALLS (COVERED CALL)
# ---------------------------------------------------
calls = df_final[df_final["tipo"] == "CALL"].copy()
calls = calls[(calls['vol'] > 0) & (calls['dias'] > 0)].copy()

T_c = calls["dias"] / 252
sigma_c = calls["vol"] / 100
d1_c = (np.log(calls["preco_atual"] / calls["strike"]) + (SELIC + sigma_c**2 / 2) * T_c) / (sigma_c * np.sqrt(T_c))
d2_c = d1_c - sigma_c * np.sqrt(T_c)

calls["black_scholes"] = (calls["preco_atual"] * norm.cdf(d1_c) - calls["strike"] * np.exp(-SELIC * T_c) * norm.cdf(d2_c)).round(2)
calls["retorno_anual"] = (calls["bid"] / calls["preco_atual"]) * (365 / calls["dias"]) * 100
calls["dist_strike"] = ((calls["strike"] / calls["preco_atual"]) - 1) * 100

filtro_call = calls[
    (calls['dist_strike'] >= 0) & (calls['retorno_anual'] >= 6) & (calls['volume'] > 0)
].copy()

if not filtro_call.empty:
    print("\n--- MELHORES OPÇÕES DE CALL COBERTA ---")
    display(filtro_call.sort_values('retorno_anual', ascending=False).head(20))


--- MELHORES OPÇÕES DE CALL COBERTA ---


,ativo,tipo,vencimento,dias,strike,symbol,bid,ask,volume,delta,theta,vol,poe,preco_atual,preco_teto,black_scholes,retorno_anual,dist_strike
1664,B3SA3,CALL,2026-04-17,9,18.56,B3SAD186,0.55,0.98,556500,0.529648,-0.042728,54.232,49.33,18.51,11.08,0.78,120.505432,0.270124
9358,PETR4,CALL,2026-04-10,4,50.00,PETRD500W2,0.60,1.50,418600,0.243606,-0.101924,54.747,22.81,48.10,34.00,0.64,113.825364,3.950104
9740,PETR4,CALL,2026-04-17,9,48.65,PETRD486,1.30,2.05,2035400,0.483281,-0.093683,44.875,45.32,48.10,34.00,1.49,109.609610,1.143451
9736,PETR4,CALL,2026-04-17,9,48.15,PETRD481,1.28,2.65,983100,0.537605,-0.094799,45.202,50.74,48.10,34.00,1.74,107.923308,0.103950
9738,PETR4,CALL,2026-04-17,9,48.40,PETRD484,1.18,3.09,1313400,0.510397,-0.094430,44.982,48.02,48.10,34.00,1.61,99.491799,0.623701
1668,B3SA3,CALL,2026-04-17,9,19.06,B3SAD191,0.45,0.00,164900,0.414163,-0.040825,52.625,37.90,18.51,11.08,0.54,98.595354,2.971367
2926,BBAS3,CALL,2026-04-10,4,23.51,BBASD234W2,0.22,0.00,132900,0.498702,-0.056391,41.773,48.17,23.43,25.00,0.48,85.680751,0.341443
10682,PETR4,CALL,2026-05-15,27,48.25,PETRE482,2.67,0.00,53300,0.564951,-0.060423,42.844,51.29,48.10,34.00,2.99,75.040425,0.311850
1666,B3SA3,CALL,2026-04-17,9,18.81,B3SAD188,0.34,0.96,267700,0.471221,-0.042162,54.615,43.51,18.51,11.08,0.67,74.494267,1.620746
1528,B3SA3,CALL,2026-04-10,4,19.31,B3SAD193W2,0.15,0.00,30200,0.265647,-0.048977,60.009,24.61,18.51,11.08,0.27,73.946515,4.321988
